In [1]:
import os
import shutil
import csv
import json
import glob

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

## Basic Utilities ====================================

In [2]:
def copy_file(source_path, destination_dir):
    """
    Copy file from sourch path to target directory. 
    Use for copying disparate datasets to centralized directory.
    """
    destination_dir = os.path.expanduser(destination_dir)
    os.makedirs(destination_dir, exist_ok=True)
    filename = os.path.basename(source_path)
    destination_path = os.path.join(destination_dir, filename)
    
    if os.path.exists(destination_path):
        print(f"{filename} already exists in {destination_dir}")
        return False
    try:
        shutil.copy2(source_path, destination_dir)
        return True
    except Exception:
        print(f"Error copying {filename}")
        return False

def move_file(source_path, destination_dir):
    """
    Move file from sourch path to target directory. 
    Use for moving disparate datasets to centralized directory.
    """
    destination_dir = os.path.expanduser(destination_dir)
    os.makedirs(destination_dir, exist_ok=True)
    filename = os.path.basename(source_path)
    destination_path = os.path.join(destination_dir, filename)

    if os.path.exists(destination_path):
        print(f"{filename} already exists in {destination_dir}")
        return False
    try:
        shutil.move(source_path, destination_dir)
        return True
    except Exception:
        print(f"Error moving {filename}")
        return False

def to_snake_case(name):
    """
    Many datasets and folders containing spaces and special characters.
    Snake case for easier handling.
    """
    name = name.replace('.', '_').replace('-', '_')
    name = ''.join(c.lower() if c.isalnum() or c == '_' else '_' for c in name)
    while '__' in name:
        name = name.replace('__', '_')
    return name.strip('_')


## Census Dataset Utilities ====================================

In [3]:
def map_census_aliases(file_path, column_mapping_dict):
    """
    Census datasets are double headered. 2 part function to handle this:
    1. Map column aliases to column codes and add dict mapping to corresponding 
    dataset dict entry.
    2. Create a multiindex pandas dataframe using codes and aliases 

    Returns:
    Multiindex pandas dataframe using codes and aliases
    """
    file_path = os.path.expanduser(file_path)
    
    # Map aliases to column codes using csv module to properly handle quoted fields
    with open(file_path, 'r', newline='', encoding='utf-8-sig') as f:
        csv_reader = csv.reader(f)
        codes = next(csv_reader)  # First row contains codes
        names = next(csv_reader)  # Second row contains names

        # Clean names
        names = [name.strip('\'"') for name in names]
        names = [' '.join(name.split()) for name in names]
        
        # Update provided dictionary directly
        column_mapping_dict.update(dict(zip(names, codes)))
    
    # Read data using column aliases
    df = pd.read_csv(file_path, skiprows=[0], encoding='utf-8-sig') 
    
    # Create MultiIndex columns with names as primary level
    df.columns = pd.MultiIndex.from_tuples(
        list(zip(codes, names)),
        names=['Code', 'Alias']
    )
    
    return df

def census_drop_cols(df, cols_to_drop):
    """
    Drop columns from multiindex dataframes based on alias header.
    Drop listed estimate columns and corresponding margin of error columns.
    Drop Empty columns.
    
    Returns:
    Multiindex pandas dataframe with estimate and margin of error columns dropped.
    """
    column_names = df.columns.names
    
    aliases = df.columns.get_level_values('Alias')
    codes = df.columns.get_level_values('Code')
    
    # Create a mapping of cleaned aliases to their original columns
    alias_to_col = {}
    for col, alias in zip(df.columns, aliases):
        # Clean alias: handle NaN and strip quotes/whitespace/colons
        clean_alias = "" if pd.isna(alias) else str(alias).strip('"').strip().rstrip(':')
        alias_to_col[clean_alias] = col
    
    # Initialize columns to drop
    columns_to_drop = set()
    
    # Drop specified columns and their margin of error pairs
    for remove_col in cols_to_drop:
        # Clean the column name we're looking for
        clean_remove_col = "" if pd.isna(remove_col) else str(remove_col).strip('"').strip().rstrip(':')
        
        # Look for exact matches in cleaned aliases
        if clean_remove_col in alias_to_col:
            columns_to_drop.add(alias_to_col[clean_remove_col])
            
            # If this is an estimate column, find and drop corresponding margin of error
            if clean_remove_col.startswith('Estimate!!'):
                margin_col = 'Margin of Error!!' + clean_remove_col[len('Estimate!!'):]
                if margin_col in alias_to_col:
                    columns_to_drop.add(alias_to_col[margin_col])
    
    # Drop only columns that have both missing/empty headers AND no values
    for col, alias, code in zip(df.columns, aliases, codes):
        # Check for missing or empty headers in either level using consistent cleaning
        clean_alias = "" if pd.isna(alias) else str(alias).strip('"').strip().rstrip(':')
        clean_code = "" if pd.isna(code) else str(code).strip('"').strip().rstrip(':')
        
        has_empty_header = (clean_alias == '' or clean_code == '')
        has_values = not df[col].isna().all()
        
        if has_empty_header and not has_values:
            columns_to_drop.add(col)
    
    # Convert set back to list for dropping
    columns_to_drop = list(columns_to_drop)
    
    # print(fr"Dropping {len(columns_to_drop)} columns ({len(columns_to_drop)//2} pairs)")
    # print(columns_to_drop)
    
    cleaned_df = df.drop(columns=columns_to_drop)
    cleaned_df.columns.names = column_names
    
    return cleaned_df

def export_census_csv(df, output_dir, filename, overwrite=False):
    """
    Export a multiindex dataframe to a single header csv (Aliases, drop Codes header).
    If overwrite, then overwrites existing files with same name, else throw error
    """
    try:
        output_path = os.path.join(os.path.expanduser(output_dir), filename)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        # Check if file exists and we're not overwriting
        if os.path.exists(output_path) and not overwrite:
            raise FileExistsError(f"File {filename} already exists and overwrite=False")
        
        # Get both column headers levels
        aliases = df.columns.get_level_values('Alias')
        cleaned_aliases = [alias.replace('"', '').replace('"', '').strip() 
                           if isinstance(alias, str) else alias for alias in aliases]

        temp_df = df.copy()
        
        # Write to csv using csv module to properly handle commas in column names
        with open(output_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            # Write only the aliases row
            writer.writerow(cleaned_aliases)
            
            temp_df.columns = cleaned_aliases
            temp_df.to_csv(f, index=False, header=False)
            
        print(f"Exported to {output_path}")
        return True
        
    except Exception as e:
        print(f"Error exporting CSV: {str(e)}")
        return False

### Dataset processing pipeline ====================================

In [4]:
def load_dataset_config(config_path, base_path=""):
    with open(os.path.expanduser(config_path), 'r') as f:
        config = json.load(f)
    
    if base_path:
        base_path = os.path.expanduser(base_path)
        for dataset in config['datasets']:
            if not os.path.isabs(dataset['original_file_path']):
                dataset['original_file_path'] = os.path.join(base_path, dataset['original_file_path'])
    
    return config['datasets']

def process_census_dataset(key, alias, original_file_path, file_blocks, central_path_head):
    centralized_file_dir = os.path.join(central_path_head, alias)
    file_blocks[key]['original_file_path'] = original_file_path
    file_blocks[key]['centralized_file_dir'] = centralized_file_dir
    
    file_name = os.path.basename(original_file_path)
    file_path = os.path.join(centralized_file_dir, file_name)
    
    cols_to_drop = file_blocks[key]['cols_to_drop']
    column_mapping_dict = file_blocks[key]['code_to_alias_column_mappings']
    
    print(f"Processing {alias}")
    
    alias_df = map_census_aliases(file_path, column_mapping_dict)    
    processed_df = census_drop_cols(alias_df, cols_to_drop)
    
    return processed_df

def load_and_process_all_datasets(config_path, base_path, file_blocks, central_path_head):
    dataset_configs = load_dataset_config(config_path, base_path)
    
    processed_datasets = {}
    
    for config in dataset_configs:
        try:
            df = process_census_dataset(
                key=config['key'],
                alias=config['alias'],
                original_file_path=config['original_file_path'],
                file_blocks=file_blocks,
                central_path_head=central_path_head
            )
            processed_datasets[config['alias']] = df
            
        except Exception as e:
            print(f"Error processing {config['alias']}: {e}")
        print("-"*30)
    
    return processed_datasets

In [5]:
path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/")
notes_wb_path = os.path.join(path_head, "Metrics/Notes/JPL_CCSVI_all_fields.xlsx")

In [6]:
notes_wb = load_workbook(notes_wb_path)
sheet = notes_wb.active

In [7]:
file_blocks = {}
current_file_name = None
cols_not_to_drop = ["Geography", "Geographic Area Name"] 

# Iterate through rows to find blocks for each csv file
for row in sheet.iter_rows(min_row=1, max_col=3, values_only=False):
    cell_value = row[0].value
    is_bold = row[0].font.bold if row[0].font else False
    
    # Detect file block by bold file name
    if is_bold and cell_value:
        original_name = cell_value.strip()
        current_file_name = to_snake_case(original_name)
        file_blocks[current_file_name] = {
            'original_name': original_name,
            'cols_to_drop': [],
            'code_to_alias_column_mappings': {},
            'original_file_path': '',
            'centralized_file_dir': '' 
        }
        continue
    
    # Check for columns with a "Subfield to keep" value
    if current_file_name and any(col.value for col in row):
        
        col_name = row[0].value
        subfield_value = row[1].value
        # Track columns with no value in the subfields to keep column
        if col_name and not subfield_value:
            if col_name not in cols_not_to_drop:
                cleaned_col_name = col_name.strip().strip('\'"')
                cleaned_col_name = ' '.join(cleaned_col_name.split())
                file_blocks[current_file_name]['cols_to_drop'].append(cleaned_col_name)

In [8]:
list(file_blocks.items())

[('age_of_structure',
  {'original_name': 'Age of Structure',
   'cols_to_drop': ['Estimate!!Total:',
    'Estimate!!Total:!!Built 2020 or later',
    'Estimate!!Total:!!Built 2010 to 2019',
    'Estimate!!Total:!!Built 2000 to 2009',
    'Estimate!!Total:!!Built 1990 to 1999'],
   'code_to_alias_column_mappings': {},
   'original_file_path': '',
   'centralized_file_dir': ''}),
 ('aggregate_number_of_vehicles_available_by_tenure',
  {'original_name': 'Aggregate number of vehicles available by tenure',
   'cols_to_drop': ['Estimate!!Aggregate number of vehicles available:!!Owner occupied',
    'Estimate!!Aggregate number of vehicles available:!!Renter occupied'],
   'code_to_alias_column_mappings': {},
   'original_file_path': '',
   'centralized_file_dir': ''}),
 ('health_insurance_coverage_by_age',
  {'original_name': 'Health insurance coverage by age',
   'cols_to_drop': ['Estimate!!Total:',
    'Estimate!!Total:!!Under 19 years:',
    'Estimate!!Total:!!Under 19 years:!!With one ty

In [9]:
# exposures_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures")
central_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/JPL")
cleaned_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/cleaned-data")
json_dir_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/census-jsons")

In [10]:
config_path = "./data/census_datasets_config.json"
base_path = "~/Desktop/Nextcloud/SCOVI Project/Metrics/"

In [11]:
datasets = load_and_process_all_datasets(config_path, base_path, file_blocks, central_path_head)

Processing age_of_structure
------------------------------
Processing aggregate_vehicles
------------------------------
Processing health_insurance
------------------------------
Processing households_w_computer
------------------------------
Processing internet_subscription
------------------------------
Processing limited_english_speaking
------------------------------
Processing living_arrangements
------------------------------
Processing income_share_of_fpl
------------------------------
Processing person_under_5_65
------------------------------
Processing population_group_quarters
------------------------------
Processing race_origin
------------------------------
Processing tenure
------------------------------
Processing 2022_census_hawaiian_homelands
------------------------------


In [ ]:
# datasets.keys()

### Age of Structure

##### JPL Notes Cleaning

In [ ]:
age_of_structure_df = datasets['age_of_structure']
age_of_structure_df.head(2)

In [ ]:
cleaned_age_of_structure_df = age_of_structure_df.iloc[:, :2].copy()

cols_to_sum = []

for col in age_of_structure_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        cols_to_sum.append(col)

total_col = ('CALCULATED', 'Total Housing Built Before 1990')
cleaned_age_of_structure_df[total_col] = age_of_structure_df[cols_to_sum].sum(axis=1)

cleaned_age_of_structure_df.head(2)

In [ ]:
export_census_csv(cleaned_age_of_structure_df, cleaned_path_head, "age_of_structure.csv", True)

### Aggregate number of vehicles

In [ ]:
aggregate_vehicles_df = datasets['aggregate_vehicles']
aggregate_vehicles_df.head(2)

In [ ]:
export_census_csv(aggregate_vehicles_df, cleaned_path_head, "aggregate_vehicles.csv", True)

### Health insurance

##### JPL Notes Cleaning

In [ ]:
health_insurance_df = datasets['health_insurance']
health_insurance_df.head(2)

In [ ]:
cleaned_health_insurance_df = health_insurance_df.iloc[:, :2].copy()

cols_to_sum = []

for col in health_insurance_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        cols_to_sum.append(col)

total_col = ('CALCULATED', 'No Health Insurance Coverage')
cleaned_health_insurance_df[total_col] = health_insurance_df[cols_to_sum].sum(axis=1)

cleaned_health_insurance_df.head(2)

In [ ]:
export_census_csv(cleaned_health_insurance_df, cleaned_path_head, "health_insurance.csv", True)

### Households with a computer

In [ ]:
households_w_computer_df = datasets['households_w_computer']
households_w_computer_df.head(2)

In [ ]:
export_census_csv(households_w_computer_df, cleaned_path_head, "households_w_computer.csv", True)

### Internet subscription

In [ ]:
internet_subscription_df = datasets['internet_subscription']
internet_subscription_df.head(2)

In [ ]:
export_census_csv(internet_subscription_df, cleaned_path_head, "internet_subscription.csv", True)

### Limited English speaking

##### JPL Notes Cleaning

In [ ]:
limited_english_speaking_df = datasets['limited_english_speaking']
limited_english_speaking_df.head(2)

In [ ]:
cols_to_sum = []

for col in limited_english_speaking_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        cols_to_sum.append(col)

cleaned_limited_english_speaking_df = limited_english_speaking_df.copy()
total_col = ('CALCULATED', 'Total Limited English Speaking Households')
cleaned_limited_english_speaking_df.insert(2, total_col, limited_english_speaking_df[cols_to_sum].sum(axis=1))

cleaned_limited_english_speaking_df.head(2)

In [ ]:
export_census_csv(cleaned_limited_english_speaking_df, cleaned_path_head, "limited_english_speaking.csv", True)

### Living Arrangements

##### JPL Notes Cleaning

In [ ]:
living_arrangements_df = datasets['living_arrangements']
living_arrangements_df.head(2)

In [ ]:
cols_to_sum = []
for col in living_arrangements_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        cols_to_sum.append(col)

cleaned_living_arrangements_df = living_arrangements_df.copy()
total_col = ('CALCULATED', r'Total "Living alone"')
cleaned_living_arrangements_df.insert(2, total_col, living_arrangements_df[cols_to_sum].sum(axis=1))
cleaned_living_arrangements_df.head(2)

In [ ]:
export_census_csv(cleaned_living_arrangements_df, cleaned_path_head, "living_arrangements.csv", True)

### Income share of FPL

##### JPL Notes Cleaning

In [ ]:
income_share_of_fpl_df = datasets['income_share_of_fpl']
income_share_of_fpl_df.head(2)

In [ ]:
cleaned_fpl_df = income_share_of_fpl_df.iloc[:, :3].copy()

under_1_cols = []
under_1_5_cols = []
under_2_cols = []

for col in income_share_of_fpl_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        col_alias = col[1]
        
        # Under 1.0
        if 'Under .50' in col_alias or '.50 to .99' in col_alias:
            under_1_cols.append(col)
            under_1_5_cols.append(col)
            under_2_cols.append(col)
        
        # 1.0 to 1.5
        elif '1.00 to 1.24' in col_alias or '1.25 to 1.49' in col_alias:
            under_1_5_cols.append(col)
            under_2_cols.append(col)
        
        # 1.5 to 2.0
        elif '1.50 to 1.84' in col_alias or '1.85 to 1.99' in col_alias:
            under_2_cols.append(col)

cleaned_fpl_df[('CALCULATED', 'Total Under 100% FPL')] = income_share_of_fpl_df[under_1_cols].sum(axis=1)
cleaned_fpl_df[('CALCULATED', 'Total Under 150% FPL')] = income_share_of_fpl_df[under_1_5_cols].sum(axis=1)
cleaned_fpl_df[('CALCULATED', 'Total Under 200% FPL')] = income_share_of_fpl_df[under_2_cols].sum(axis=1)

cleaned_fpl_df.head(3)

In [ ]:
export_census_csv(cleaned_fpl_df, cleaned_path_head, "income_share_of_fpl.csv", True)

### Persons under 5 & 65

##### JPL Notes Cleaning

In [ ]:
person_under_5_65_df = datasets['person_under_5_65']
person_under_5_65_df.head(2)

In [ ]:
cleaned_genders_df = person_under_5_65_df.iloc[:, :2].copy()
cleaned_genders_df[('B01001_002E', 'Estimate!!Total:!!Male:')] = person_under_5_65_df[('B01001_002E', 'Estimate!!Total:!!Male:')]
cleaned_genders_df[('B01001_026E', 'Estimate!!Total:!!Female:')] = person_under_5_65_df[('B01001_026E', 'Estimate!!Total:!!Female:')]

cleaned_genders_df['CALCULATED', 'Total_Population'] = cleaned_genders_df[('B01001_002E', 'Estimate!!Total:!!Male:')] + cleaned_genders_df[('B01001_026E', 'Estimate!!Total:!!Female:')]

cleaned_genders_df.head(2)

In [ ]:
export_census_csv(cleaned_genders_df, cleaned_path_head, "genders.csv", True)

In [ ]:
cleaned_males_df = person_under_5_65_df.iloc[:, :2].copy()

males_under_5_cols = []
males_under_18_cols = []
males_over_65_cols = []

for col in person_under_5_65_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!Male:!!'):
        col_alias = col[1]
        
        # Under 5
        if 'Under 5' in col_alias:
            males_under_5_cols.append(col)
            males_under_18_cols.append(col)
        
        # Under 18
        elif ('5 to 9' in col_alias or '10 to 14' in col_alias or 
              '15 to 17' in col_alias):
            males_under_18_cols.append(col)
        
        # 65 and over
        elif ('65 and 66' in col_alias or '67 to 69' in col_alias or
              '70 to 74' in col_alias or '75 to 79' in col_alias or
              '80 to 84' in col_alias or '85' in col_alias):
            males_over_65_cols.append(col)

cleaned_males_df[('CALCULATED', 'Males Under 5')] = person_under_5_65_df[males_under_5_cols].sum(axis=1)
cleaned_males_df[('CALCULATED', 'Males Under 18')] = person_under_5_65_df[males_under_18_cols].sum(axis=1)
cleaned_males_df[('CALCULATED', 'Males Over 65')] = person_under_5_65_df[males_over_65_cols].sum(axis=1)

cleaned_males_df.head(2)

In [ ]:
export_census_csv(cleaned_males_df, cleaned_path_head, "person_under_5_65_males.csv", True)

In [ ]:
cleaned_females_df = person_under_5_65_df.iloc[:, :2].copy()

females_under_5_cols = []
females_under_18_cols = []
females_over_65_cols = []

for col in person_under_5_65_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!Female:!!'):
        col_alias = col[1]
        
        # Under 5
        if 'Under 5' in col_alias:
            females_under_5_cols.append(col)
            females_under_18_cols.append(col)
        
        # Under 18
        elif ('5 to 9' in col_alias or '10 to 14' in col_alias or 
              '15 to 17' in col_alias):
            females_under_18_cols.append(col)
        
        # 65 and over
        elif ('65 and 66' in col_alias or '67 to 69' in col_alias or
              '70 to 74' in col_alias or '75 to 79' in col_alias or
              '80 to 84' in col_alias or '85' in col_alias):
            females_over_65_cols.append(col)

cleaned_females_df[('CALCULATED', 'Females Under 5')] = person_under_5_65_df[females_under_5_cols].sum(axis=1)
cleaned_females_df[('CALCULATED', 'Females Under 18')] = person_under_5_65_df[females_under_18_cols].sum(axis=1)
cleaned_females_df[('CALCULATED', 'Females Over 65')] = person_under_5_65_df[females_over_65_cols].sum(axis=1)

cleaned_females_df.head(2)

In [ ]:
export_census_csv(cleaned_females_df, cleaned_path_head, "person_under_5_65_females.csv", True)

### Population in group quarters

In [ ]:
population_group_quarters_df = datasets['population_group_quarters']
population_group_quarters_df.head(2)

In [ ]:
export_census_csv(population_group_quarters_df, cleaned_path_head, "population_group_quarters.csv", True)

### Race origin

In [ ]:
race_origin_df = datasets['race_origin']
race_origin_df.head(2)

In [ ]:
export_census_csv(race_origin_df, cleaned_path_head, "race_origin.csv", True)

### Tenure

In [ ]:
tenure_df = datasets['tenure']
tenure_df.head(2)

In [ ]:
export_census_csv(tenure_df, cleaned_path_head, "tenure.csv", True)

### 2022 Census Hawaiian Homelands

In [ ]:
tenure_df = datasets['tenure']
tenure_df.head(2)

In [ ]:
# export_census_csv(tenure_df, cleaned_path_head, "2022_census_hawaiian_homelands.csv", True)

##### JPL Notes Cleaning

In [12]:
hawaiian_homelands_df = datasets['2022_census_hawaiian_homelands']
hawaiian_homelands_df.head(2)

Code,GEO_ID,NAME,S0601_C01_002E,S0601_C01_002M,S0601_C01_003E,S0601_C01_003M,S0601_C01_008E,S0601_C01_008M,S0601_C01_009E,S0601_C01_009M,...,S0601_C01_022E,S0601_C01_022M,S0601_C01_026E,S0601_C01_026M,S0601_C01_047E,S0601_C01_047M,S0601_C01_049E,S0601_C01_049M,S0601_C01_050E,S0601_C01_050M
Alias,Geography,Geographic Area Name,Estimate!!Total!!Total population!!AGE!!Under 5 years,Margin of Error!!Total!!Total population!!AGE!!Under 5 years,Estimate!!Total!!Total population!!AGE!!5 to 17 years,Margin of Error!!Total!!Total population!!AGE!!5 to 17 years,Estimate!!Total!!Total population!!AGE!!65 to 74 years,Margin of Error!!Total!!Total population!!AGE!!65 to 74 years,Estimate!!Total!!Total population!!AGE!!75 years and over,Margin of Error!!Total!!Total population!!AGE!!75 years and over,...,"Estimate!!Total!!Total population!!RACE AND HISPANIC OR LATINO ORIGIN!!White alone, not Hispanic or Latino","Margin of Error!!Total!!Total population!!RACE AND HISPANIC OR LATINO ORIGIN!!White alone, not Hispanic or Latino",Estimate!!Total!!LANGUAGE SPOKEN AT HOME AND ABILITY TO SPEAK ENGLISH!!Population 5 years and over!!Speak language other than English!!Speak English less than very well,Margin of Error!!Total!!LANGUAGE SPOKEN AT HOME AND ABILITY TO SPEAK ENGLISH!!Population 5 years and over!!Speak language other than English!!Speak English less than very well,Estimate!!Total!!INDIVIDUALS' INCOME IN THE PAST 12 MONTHS (IN 2022 INFLATION-ADJUSTED DOLLARS)!!Population 15 years and over!!Median income (dollars),Margin of Error!!Total!!INDIVIDUALS' INCOME IN THE PAST 12 MONTHS (IN 2022 INFLATION-ADJUSTED DOLLARS)!!Population 15 years and over!!Median income (dollars),Estimate!!Total!!POVERTY STATUS IN THE PAST 12 MONTHS!!Population for whom poverty status is determined!!Below 100 percent of the poverty level,Margin of Error!!Total!!POVERTY STATUS IN THE PAST 12 MONTHS!!Population for whom poverty status is determined!!Below 100 percent of the poverty level,Estimate!!Total!!POVERTY STATUS IN THE PAST 12 MONTHS!!Population for whom poverty status is determined!!100 to 149 percent of the poverty level,Margin of Error!!Total!!POVERTY STATUS IN THE PAST 12 MONTHS!!Population for whom poverty status is determined!!100 to 149 percent of the poverty level
0,2500000US5003,"Anahola (Agricultural) Hawaiian Home Land, HI",16.9,9.7,22.6,14.7,15.2,9.5,6.4,6.7,...,16.9,7.0,0.0,12.4,24792,8399,10.8,12.3,29.1,33.5
1,2500000US5004,"Anahola (Residential) Hawaiian Home Land, HI",6.3,2.5,18.1,4.2,9.8,3.0,9.2,2.9,...,7.3,3.1,1.6,1.3,32705,3044,11.4,4.4,6.5,4.1


In [13]:
cleaned_hawaiian_homelands_df = hawaiian_homelands_df.iloc[:, :2].copy()

under_5_cols = []
under_18_cols = []
over_65_cols = []

for col in hawaiian_homelands_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total!!Total population!!'):
        col_alias = col[1]
        
        # Under 5
        if 'Under 5' in col_alias:
            under_5_cols.append(col)
            under_18_cols.append(col)
        
        # Under 18
        elif '5 to 17' in col_alias:
            under_18_cols.append(col)
        
        # 65 and over
        elif ('65 to 74' in col_alias or '75' in col_alias):
            over_65_cols.append(col)

cleaned_hawaiian_homelands_df[('CALCULATED', 'Total Population Under 5')] = hawaiian_homelands_df[under_5_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
cleaned_hawaiian_homelands_df[('CALCULATED', 'Total Population Under 18')] = hawaiian_homelands_df[under_18_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
cleaned_hawaiian_homelands_df[('CALCULATED', 'Total Population Over 65')] = hawaiian_homelands_df[over_65_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)

cleaned_hawaiian_homelands_df.head(2)

Code          GEO_ID                                           NAME  \
Alias      Geography                           Geographic Area Name   
0      2500000US5003  Anahola (Agricultural) Hawaiian Home Land, HI   
1      2500000US5004   Anahola (Residential) Hawaiian Home Land, HI   

Code                CALCULATED                            \
Alias Total Population Under 5 Total Population Under 18   
0                         16.9                      39.5   
1                          6.3                      24.4   

Code                            
Alias Total Population Over 65  
0                         21.6  
1                         19.0

In [14]:
cleaned_hawaiian_homelands_df = pd.concat([cleaned_hawaiian_homelands_df, hawaiian_homelands_df.iloc[:, 10:-4].copy()], axis=1)
cleaned_hawaiian_homelands_df.head(2)

Code          GEO_ID                                           NAME  \
Alias      Geography                           Geographic Area Name   
0      2500000US5003  Anahola (Agricultural) Hawaiian Home Land, HI   
1      2500000US5004   Anahola (Residential) Hawaiian Home Land, HI   

Code                CALCULATED                            \
Alias Total Population Under 5 Total Population Under 18   
0                         16.9                      39.5   
1                          6.3                      24.4   

Code                                                         S0601_C01_011E  \
Alias Total Population Over 65 Estimate!!Total!!Total population!!SEX!!Male   
0                         21.6                                         43.9   
1                         19.0                                         48.2   

Code                                       S0601_C01_011M  \
Alias Margin of Error!!Total!!Total population!!SEX!!Male   
0                                                   10.9    
1                                                    4.5    

Code                                  S0601_C01_012E  \
Alias Estimate!!Total!!Total population!!SEX!!Female   
0                                               56.1   
1                                               51.8   

Code                                         S0601_C01_012M  \
Alias Margin of Error!!Total!!Total population!!SEX!!Female   
0                                                   10.9      
1                                                    4.5      

Code                                                                          S0601_C01_014E  \
Alias Estimate!!Total!!Total population!!RACE AND HISPANIC OR LATINO ORIGIN!!One race!!White   
0                                                   16.9                                       
1                                                    7.4                                       

Code   ...  \
Alias  ...   
0      ...   
1      ...   

Code                                                                            S0601_C01_020E  \
Alias Estimate!!Total!!Total population!!RACE AND HISPANIC OR LATINO ORIGIN!!Two or more races   
0                                                   52.4                                         
1                                                   29.8                                         

Code                                                                                   S0601_C01_020M  \
Alias Margin of Error!!Total!!Total population!!RACE AND HISPANIC OR LATINO ORIGIN!!Two or more races   
0                                                   18.6                                                
1                                                    6.9                                                

Code                                                                                                  S0601_C01_021E  \
Alias Estimate!!Total!!Total population!!RACE AND HISPANIC OR LATINO ORIGIN!!Hispanic or Latino origin (of any race)   
0                                                    0.3                                                               
1                                                    7.7                                                               

Code                                                                                                         S0601_C01_021M  \
Alias Margin of Error!!Total!!Total population!!RACE AND HISPANIC OR LATINO ORIGIN!!Hispanic or Latino origin (of any race)   
0                                                    1.2                                                                      
1                                                    4.1                                                                      

Code                                                                                              S0601_C01_022E  \
Alias Estimate!!Total!!Total population!!RACE AND HISPANIC OR LATINO ORI

In [ ]:
export_census_csv(cleaned_hawaiian_homelands_df, cleaned_path_head, "2022_census_hawaiian_homelands.csv", True)

##### Add Hawaiian Homelands census poverty data to FPL poverty status dataset

In [ ]:
cleaned_fpl_df = pd.concat([cleaned_fpl_df, hawaiian_homelands_df.iloc[:, -4:].copy()], axis=1)
cleaned_fpl_df.head(2)

In [ ]:
export_census_csv(cleaned_fpl_df, cleaned_path_head, "income_share_of_fpl.csv", True)

## ============================================================

In [ ]:
# import glob

# csv_files = glob.glob(os.path.join(cleaned_path_head, "*.csv"))

# # Process each CSV file
# for csv_file in csv_files:
#     print(f"Processing {csv_file}...")
#     try:
#         census_csv_to_json(csv_file, json_dir_path)
#     except Exception as e:
#         print(f"Error processing {csv_file}: {e}")

## Leaflet JSON 

In [8]:
def clean_column_name(col_name, prefixes_to_remove):
    """
    Remove specified prefixes from column name and clean up remaining !! separators and trailing colons
    """
    # Remove prefixes
    for prefix in prefixes_to_remove:
        if col_name.startswith(prefix):
            col_name = col_name[len(prefix):]
            break
    
    cleaned_name = col_name.replace('!!', ' ')
    cleaned_name = ' '.join(cleaned_name.split())
    cleaned_name = cleaned_name.rstrip(':')
    
    return cleaned_name

# This version processes multiple csvs into a single master json
def census_csvs_to_master_json(csv_directory, json_path):
    """
    Process all census CSV files of a given directory into a single master JSON file.
    """       
    csv_directory = os.path.expanduser(csv_directory)
    json_path = os.path.expanduser(json_path)
    
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    
    csv_files = glob.glob(os.path.join(csv_directory, "*.csv"))
    
    metrics = {}
    prefixes_to_remove = ['Estimate!!Total:!!', 'Estimate!!Total!!', 'Margin of Error!!', '!!Total:!!']
    
    for csv_file in csv_files:
        csv_filename = os.path.basename(csv_file)
        census_metric_name = os.path.splitext(csv_filename)[0]
        
        try:
            na_values = ["", "-", "**", "null"]
            df = pd.read_csv(csv_file, na_values=na_values)
            
            for _, row in df.iterrows():
                geo_id = row['Geography']
                # Remove the "1500000US" prefix
                if isinstance(geo_id, str) and 'US' in geo_id:
                    geo_id = geo_id.split('US')[1]
                
                geo_name = row['Geographic Area Name']
                
                if geo_id not in metrics:
                    # Split geographic name
                    geo_name_parts = [geo_name_part.strip() for geo_name_part in geo_name.split(';')]
                    
                    # Handles Hawaiian Homelands data that does not follow typical geo area name structure
                    if len(geo_name_parts) < 4:
                        metrics[geo_id] = {
                            "block_group": geo_name,
                            "census_tract": None,
                            "county": None,
                            "state": None,
                            "metrics": {}
                        }
                    else:
                        metrics[geo_id] = {
                            "block_group": geo_name_parts[0],
                            "census_tract": geo_name_parts[1],
                            "county": geo_name_parts[2],
                            "state": geo_name_parts[3],
                            "metrics": {}
                        }

                # Group metrics by csv
                if census_metric_name not in metrics[geo_id]["metrics"]:
                    metrics[geo_id]["metrics"][census_metric_name] = {}
                
                # Add all estimate columns
                for col in df.columns:
                    if col == 'Geography' or col == 'Geographic Area Name' or col.startswith('Margin of Error'):
                        continue
                    
                    # Clean column name by removing prefixes
                    field_name = clean_column_name(col, prefixes_to_remove)
                    
                    if pd.isna(row[col]):
                        metrics[geo_id]["metrics"][census_metric_name][field_name] = None
                        continue
                                            
                    value = int(row[col])
                    
                    metrics[geo_id]["metrics"][census_metric_name][field_name] = value
                    
        except Exception as e:
            print(f"Error processing {csv_file}: {e}")
            continue
    
    # Write to JSON file
    with open(json_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    print(f"JSON at {json_path}")
    return metrics

def generate_dataset_params(csv_directory, json_path):
    csv_directory = os.path.expanduser(csv_directory)
    json_path = os.path.expanduser(json_path)
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    
    csv_files = glob.glob(os.path.join(csv_directory, "*.csv"))
    
    dataset_params = {}
    prefixes_to_remove = ['Estimate!!Total:!!', 'Estimate!!Total!!', 'Margin of Error!!', '!!Total:!!']
    
    for csv_file in csv_files:
        csv_filename = os.path.basename(csv_file)
        key = os.path.splitext(csv_filename)[0]

        hawaiian_homelands = "hawaiian_homelands" in csv_filename.lower()
        
        dataset_params[key] = {
            'metricName': '',
            'metricLabel': '',
            'hawaiianHomelands': hawaiian_homelands,
            'columnThresholds': {},
        }
        
        na_values = ["", "-", "**", "null"]
        df = pd.read_csv(csv_file, na_values=na_values)
        
        for col in df.columns:
            if col == 'Geography' or col == 'Geographic Area Name' or col.startswith('Margin of Error'):
                continue

            # Clean column name by removing prefixes
            field_name = clean_column_name(col, prefixes_to_remove)
            
            # This section calculates quantiles using pandas qcut rounded to nearest factor of 5
            numeric_col = pd.to_numeric(df[col])
            
            labels, edges = pd.qcut(numeric_col, q=10, labels=False, retbins=True, duplicates='drop')
            
            # Round edges to nearest factor of 5 directly in the loop
            rounded_edges = [5 * round(edge/5) for edge in edges]

            dataset_params[key]['columnThresholds'][field_name] = {
                'thresholds': rounded_edges,
                'colors': ['#FFEDA0', '#FED976', '#FEB24C', '#FD8D3C', '#FC4E2A', '#E31A1C', '#BD0026', '#800026', '#5A0018', '#3A000F']
            }
            
    with open(json_path, 'w') as f:
        json.dump(dataset_params, f, indent=2)
        
    return dataset_params

In [9]:
json_output_path = os.path.join(json_dir_path,"all_census.json") 

metrics = census_csvs_to_master_json(cleaned_path_head, json_output_path)

JSON at /Users/andyyu/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/census-jsons/all_census.json


In [10]:
# cleaned_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/cleaned-data")

# json_dir_path = os.path.expanduser("~/Desktop")
json_output_path = os.path.join(json_dir_path,"census_datasets_info.json") 

dataset_params = generate_dataset_params(cleaned_path_head, json_output_path)